In [1]:
!pip install python-dotenv
!pip install openai

# TRATAMIENTO DE UNIDADES DE OBRA DE PROYECTOS

In [10]:
# -*- coding: utf-8 -*-
"""
Procesador de textos con salida estructurada y validación.
- Edita libremente la lista `ITEMS` y las PAUTAS en PROMPT_TEMPLATE.
- Usa response_format JSON para garantizar estructura: {"item_key", "texto_alterado"}.
- Valida cada salida antes de almacenarla.
- Guarda resultados en JSONL y CSV.

Requisitos:
  pip install openai python-dotenv
  export OPENAI_API_KEY="tu_clave"
o crea un .env con OPENAI_API_KEY=tu_clave
"""

import os
import json
import csv
import time
from typing import Dict, List, Any, Optional
import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI, OpenAIError

# =========================
#  Configuración
# =========================
load_dotenv()  # carga .env si existe
API_KEY = os.getenv("OPENAI_API_KEY")
if not API_KEY:
    raise RuntimeError("Falta OPENAI_API_KEY en el entorno o .env")

MODEL = "gpt-4o-mini"  # cámbialo si prefieres otro modelo

# =========================
#  Datos de entrada
# =========================
ITEMS: List[Dict[str, str]] = [
    {
        "item_key": "item1",
        "texto": (
            "Zanja para tendido de cables de 1,10 m de profundidad y 0,60 m de anchura máxima, "
            "zona de balasto, ejecutada a mano incluso descerne y entibación de costados y posterior "
            "reposición del balasto retirado, y relleno compactado con material seleccionado procedente "
            "de la excavación, cama de arena de 40 cm de espesor, rejilla de plástico identificadora, "
            "carga, descarga y transporte al lugar asignado en el interior de la obra.\n"
            "Trabajo: Cualquier franja horaria\n"
            "Banda de mantenimiento: No necesita intervalo\n"
            "Condiciones de ejecución: Cualquier condición de ejecución"
        )
    },
    {
        "item_key": "item2", 
        "texto":( 
            "Zanja para tendido de cables de hasta 0,80 m de profundidad y 0,60 m de anchura máxima, "
            "ejecutada por excavación manual en cualquier tipo de terreno excepto roca, y relleno compactado "
            "con material seleccionado procedente de la excavación, cama de arena de 20 cm de espesor, "
            "rejilla de plastico identificadora, carga, descarga y transporte al lugar asignado en el interior "
            "de la obra.\n"
            "Trabajo: Diurno\n"
            "Banda de mantenimiento: i >= 5 horas\n"
            "Condiciones de ejecución: Volumen relevante\n"
         )
    },
    {
        "item_key": "item3", 
        "texto":(
            "Canalización hormigonada de 1 tubos de polietileno libre de halógenos "
            "de 110 mm de diámetro en cualquier clase de terreno, excepto roca, "
            "incluso  el relleno y compactado de la zanja,  el suministro y montaje "
            "de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, "
            "el transporte y la retirada de los productos al lugar de empleo/n" 
            "Trabajo: Diurno/n"
            "Banda de mantenimiento: i >= 5 horas/n"
            "Condiciones de ejecución: Volumen relevante/n"
        )
    }
]

# =========================
#  Plantilla con PAUTAS
# =========================
PROMPT_TEMPLATE = (
    "Transforma el texto siguiendo estrictamente estas pautas:\n"
    "1) Mantén el significado original y todas las palabras clave.\n"
    "2) Desarrolla el texto añadiendo detalles constructivos adicionales sin alterar la naturaleza ni el dimensionamiento.\n\n"
    "Ejemplo:\n"
    "texto a transformar:\n"
    "Zanja para tendido de cables de 1,10 m de profundidad y 0,60 m de anchura máxima, zona de balasto, "
    "ejecutada a mano incluso descerne y entibación de costados y posterior reposición del balasto retirado, "
    "y relleno compactado con material seleccionado procedente de la excavación, cama de arena de 40 cm de espesor, "
    "rejilla de plastico identificadora, carga, descarga y transporte al lugar asignado en el interior de la obra.\n"
    "Trabajo: Cualquier franja horaria\n"
    "Banda de mantenimiento: No necesita intervalo\n"
    "Condiciones de ejecución: Cualquier condición de ejecución.\n\n"
    "texto transformado:\n"
    "ZANJA PARA TENDIDO DE CABLES DE 1,10 m DE PROFUNDIDAD Y 0,60 m DE ANCHURA MÁXIMA EN ZONA DE BALASTO, "
    "INCLUSO DESCERNE Y ENTIBACIÓN DE COSTADOS Y POSTERIOR REPOSICIÓN DEL BALASTO RETIRADO, EXCAVACIÓN MANUAL "
    "Y RELLENO COMPACTADO CON MATERIAL SELECCIONADO PROCEDENTE DE LA EXCAVACIÓN, CAMA DE ARENA DE RÍO DE 20 cm "
    "DE ESPESOR, REJILLA DE PLÁSTICO IDENTIFICADORA, CARGA, TRANSPORTE Y DESCARGA DE LOS PRODUCTOS EN SU LUGAR DE EMPLEO.\n"
    "Trabajo: Cualquier franja horaria\n"
    "Banda de mantenimiento: No necesita intervalo\n"
    "Condiciones de ejecución: Cualquier condición de ejecución.\n\n"
    "Devuelve la respuesta ÚNICAMENTE como un JSON con esta estructura EXACTA:\n"
    '{{"item_key":"{item_key}", "texto_alterado":"<texto_transformado>"}}\n'
    "donde <texto_transformado> es el texto final. Máx. {max_palabras} palabras.\n\n"
    "Texto a transformar:\n{texto}"
)

# Parámetros de validación
MAX_PALABRAS = 500
MIN_CARACTERES = 3

# =========================
#  Cliente OpenAI
# =========================
client = OpenAI(api_key=API_KEY)

# =========================
#  Utilidades
# =========================
def build_prompt(item_key: str, texto: str, max_palabras: int = MAX_PALABRAS) -> str:
    return PROMPT_TEMPLATE.format(item_key=item_key, texto=texto, max_palabras=max_palabras)

def parse_json_strict(s: str) -> Dict[str, Any]:
    """
    Intenta convertir la cadena en JSON. Lanza ValueError con mensaje claro si falla.
    """
    try:
        return json.loads(s)
    except json.JSONDecodeError as e:
        # A veces los modelos devuelven comillas inteligentes u otros artefactos: intentar limpiar mínimo
        s2 = s.strip().strip("```").strip()
        try:
            return json.loads(s2)
        except Exception:
            raise ValueError(f"Salida no es JSON válido. Contenido recibido:\n{s}\nError: {e}")

def validar_salida(item_key_esperado: str, texto_alterado: str) -> Optional[str]:
    """
    Devuelve None si todo OK; string con motivo si hay error.
    """
    if not texto_alterado or not texto_alterado.strip():
        return "texto_alterado vacío."
    if len(texto_alterado.strip()) < MIN_CARACTERES:
        return f"texto_alterado demasiado corto (<{MIN_CARACTERES} caracteres)."
    # Validar conteo de palabras
    num_palabras = len(texto_alterado.split())
    if num_palabras > MAX_PALABRAS:
        return f"Excede el máximo de {MAX_PALABRAS} palabras (tiene {num_palabras})."
    # Validación adicional opcional: evitar prefijos tipo "Texto reescrito:" o llaves
    prohibidos = ["Texto reescrito:", "```", "{", "}", "[", "]"]
    if any(tok in texto_alterado[:20] for tok in prohibidos):  # heurística
        # Permitimos llaves si forman parte normal del español, pero aquí suelen indicar salida no limpia
        return "Salida parece contener formato o metatexto no deseado al inicio."
    return None

def llamar_openai_estructurado(prompt: str, max_reintentos: int = 3, pausa_base: float = 1.0) -> Dict[str, Any]:
    """
    Llama a OpenAI pidiendo JSON estricto. Reintenta con backoff si falla por rate/temporales.
    """
    intento = 0
    while True:
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                response_format={"type": "json_object"},
            )
            contenido = resp.choices[0].message.content
            data = parse_json_strict(contenido)
            return data
        except (OpenAIError, ValueError) as e:
            intento += 1
            if intento > max_reintentos:
                raise
            time.sleep(pausa_base * (2 ** (intento - 1)))

# =========================
#  Proceso principal
# =========================
def procesar_items(items: List[Dict[str, str]]) -> List[Dict[str, str]]:
    resultados: List[Dict[str, str]] = []
    errores: List[Dict[str, str]] = []

    for item in items:
        item_key = item["item_key"]
        texto = item["texto"]

        prompt = build_prompt(item_key=item_key, texto=texto)
        try:
            data = llamar_openai_estructurado(prompt)
            # Esperamos {"item_key": "...", "texto_alterado": "..."}
            if "item_key" not in data or "texto_alterado" not in data:
                raise ValueError(f"Faltan claves en la salida: {data}")

            if str(data["item_key"]) != str(item_key):
                raise ValueError(f"item_key devuelto ({data['item_key']}) no coincide con el esperado ({item_key}).")

            texto_alterado: str = str(data["texto_alterado"]).strip()

            # Validaciones en código
            motivo = validar_salida(item_key, texto_alterado)
            if motivo:
                errores.append({"item_key": item_key, "motivo": motivo})
                print(f"⚠️ {item_key}: {motivo}")
                continue

            resultados.append({"item_key": item_key, "texto_alterado": texto_alterado})

        except Exception as e:
            errores.append({"item_key": item_key, "motivo": f"Excepción: {e}"})
            print(f"❌ {item_key}: {e}")

    # Persistencia
    if resultados:
        with open("resultados.jsonl", "w", encoding="utf-8") as f:
            for r in resultados:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        with open("resultados.csv", "w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=["item_key", "texto_alterado"])
            writer.writeheader()
            writer.writerows(resultados)

    # Reporte pequeño
    print("\n=== RESUMEN ===")
    print(f"Procesados: {len(items)} | OK: {len(resultados)} | Errores: {len(errores)}")
    if errores:
        for e in errores:
            print(f"- {e['item_key']}: {e['motivo']}")

    return resultados

In [11]:
resultados = procesar_items(ITEMS)


=== RESUMEN ===
Procesados: 3 | OK: 3 | Errores: 0


In [19]:
# Convertir a DataFrames
df_items = pd.DataFrame(ITEMS)
df_resultados = pd.DataFrame(resultados)

# Unir por item_key
df_final = pd.merge(df_items, df_resultados, on="item_key", how="inner")

# Visualización simple por consola
print("\n=== RESULTADOS ===")
pd.set_option("display.max_colwidth", None)   # no truncar celdas
pd.set_option("display.width", 0)            # ajuste automático del ancho
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
df_final


=== RESULTADOS ===


,item_key,texto,texto_alterado
0,item1,"Zanja para tendido de cables de 1,10 m de profundidad y 0,60 m de anchura máxima, zona de balasto, ejecutada a mano incluso descerne y entibación de costados y posterior reposición del balasto retirado, y relleno compactado con material seleccionado procedente de la excavación, cama de arena de 40 cm de espesor, rejilla de plástico identificadora, carga, descarga y transporte al lugar asignado en el interior de la obra.\nTrabajo: Cualquier franja horaria\nBanda de mantenimiento: No necesita intervalo\nCondiciones de ejecución: Cualquier condición de ejecución","ZANJA PARA TENDIDO DE CABLES DE 1,10 m DE PROFUNDIDAD Y 0,60 m DE ANCHURA MÁXIMA EN ZONA DE BALASTO, EJECUTADA A MANO, INCLUYENDO DESCERNE Y ENTIBACIÓN DE COSTADOS, ASÍ COMO LA POSTERIOR REPOSICIÓN DEL BALASTO RETIRADO. LA EXCAVACIÓN SE REALIZARÁ MANUALMENTE, GARANTIZANDO UN RELLENO COMPACTADO CON MATERIAL SELECCIONADO PROCEDENTE DE LA EXCAVACIÓN, LO QUE ASEGURARÁ LA ESTABILIDAD Y DURABILIDAD DE LA ZANJA. SE INCLUIRÁ UNA CAMA DE ARENA DE RÍO DE 40 cm DE ESPESOR, QUE SERVIRÁ COMO BASE PARA PROTEGER LOS CABLES Y MEJORAR LA DRENAJE DEL AGUA. SE COLOCARÁ UNA REJILLA DE PLÁSTICO IDENTIFICADORA PARA MARCAR LA ZONA Y EVITAR ACCIDENTES FUTUROS. EL PROCESO INCLUYE CARGA, TRANSPORTE Y DESCARGA DE LOS MATERIALES EN SU LUGAR DE EMPLEO DENTRO DEL ÁREA ASIGNADA EN LA OBRA, ASEGURANDO QUE TODO SE REALICE DE MANERA ORDENADA Y EFICAZ. \nTrabajo: Cualquier franja horaria\nBanda de mantenimiento: No necesita intervalo\nCondiciones de ejecución: Cualquier condición de ejecución."
1,item2,"Zanja para tendido de cables de hasta 0,80 m de profundidad y 0,60 m de anchura máxima, ejecutada por excavación manual en cualquier tipo de terreno excepto roca, y relleno compactado con material seleccionado procedente de la excavación, cama de arena de 20 cm de espesor, rejilla de plastico identificadora, carga, descarga y transporte al lugar asignado en el interior de la obra.\nTrabajo: Diurno\nBanda de mantenimiento: i >= 5 horas\nCondiciones de ejecución: Volumen relevante\n","ZANJA PARA TENDIDO DE CABLES DE HASTA 0,80 m DE PROFUNDIDAD Y 0,60 m DE ANCHURA MÁXIMA, EJECUTADA POR EXCAVACIÓN MANUAL EN CUALQUIER TIPO DE TERRENO EXCEPTO ROCA, LO QUE PERMITE UNA MAYOR FLEXIBILIDAD EN LA EJECUCIÓN DEL TRABAJO. EL RELLENO SERÁ COMPACTADO CON MATERIAL SELECCIONADO PROCEDENTE DE LA EXCAVACIÓN, GARANTIZANDO ASÍ LA ESTABILIDAD Y DURABILIDAD DE LA ZANJA. SE INCLUIRÁ UNA CAMA DE ARENA DE 20 cm DE ESPESOR, QUE ACTUARÁ COMO BASE PARA PROTEGER LOS CABLES Y MEJORAR LA DISTRIBUCIÓN DE CARGAS. SE COLOCARÁ UNA REJILLA DE PLÁSTICO IDENTIFICADORA PARA SEÑALIZAR LA ZONA Y EVITAR ACCIDENTES DURANTE Y DESPUÉS DE LA EJECUCIÓN DE LOS TRABAJOS. EL PROCESO INCLUYE CARGA, DESCARGA Y TRANSPORTE DE LOS MATERIALES AL LUGAR ASIGNADO EN EL INTERIOR DE LA OBRA, ASEGURANDO UNA ORGANIZACIÓN EFICAZ Y UN FLUJO DE TRABAJO SIN INTERRUPCIONES. \nTrabajo: Diurno\nBanda de mantenimiento: i >= 5 horas\nCondiciones de ejecución: Volumen relevante."
2,item3,"Canalización hormigonada de 1 tubos de polietileno libre de halógenos de 110 mm de diámetro en cualquier clase de terreno, excepto roca, incluso el relleno y compactado de la zanja, el suministro y montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, el transporte y la retirada de los productos al lugar de empleo/nTrabajo: Diurno/nBanda de mantenimiento: i >= 5 horas/nCondiciones de ejecución: Volumen relevante/n","CANALIZACIÓN HORMIGONADA DE 1 TUBOS DE POLIETILENO LIBRE DE HALÓGENOS DE 110 mm DE DIÁMETRO EN CUALQUIER CLASE DE TERRENO, EXCEPTO ROCA, INCLUYENDO EL RELLENO Y COMPACTADO DE LA ZANJA, ASÍ COMO EL SUMINISTRO Y MONTAJE DE LOS TUBOS Y HORMIGÓN TIPO HE-20 SIN VIBRAR. SE REALIZARÁN PRUEBAS DE LOS CONDUCTOS PARA GARANTIZAR SU INTEGRIDAD Y FUNCIONALIDAD, Y SE LLEVARÁ A CABO EL TRANSPORTE Y LA RETIRADA DE LOS PRODUCTOS AL LUGAR DE EMPLEO, ASEGURANDO QUE CADA ELEMENTO SEA MANIPULADO CON CUIDADO PARA EVITAR DAÑOS. \nTrabajo

# TRATAMIENTO DE UNIDADES DE OBRA DE PARTES DE TRABAJO

In [20]:
# =========================
#  Datos de entrada
# =========================
ITEMS: List[Dict[str, str]] = [
    {
        "item_key": "item1",
        "texto": (
            "Zanja para tendido de cables de 1,10 m de profundidad y 0,60 m de anchura máxima, "
            "zona de balasto, ejecutada a mano incluso descerne y entibación de costados y posterior "
            "reposición del balasto retirado, y relleno compactado con material seleccionado procedente "
            "de la excavación, cama de arena de 40 cm de espesor, rejilla de plástico identificadora, "
            "carga, descarga y transporte al lugar asignado en el interior de la obra.\n"
            "Trabajo: Cualquier franja horaria\n"
            "Banda de mantenimiento: No necesita intervalo\n"
            "Condiciones de ejecución: Cualquier condición de ejecución"
        )
    },
    {
        "item_key": "item2", 
        "texto":( 
            "Zanja para tendido de cables de hasta 0,80 m de profundidad y 0,60 m de anchura máxima, "
            "ejecutada por excavación manual en cualquier tipo de terreno excepto roca, y relleno compactado "
            "con material seleccionado procedente de la excavación, cama de arena de 20 cm de espesor, "
            "rejilla de plastico identificadora, carga, descarga y transporte al lugar asignado en el interior "
            "de la obra.\n"
            "Trabajo: Diurno\n"
            "Banda de mantenimiento: i >= 5 horas\n"
            "Condiciones de ejecución: Volumen relevante\n"
         )
    },
    {
        "item_key": "item3", 
        "texto":(
            "Canalización hormigonada de 1 tubos de polietileno libre de halógenos "
            "de 110 mm de diámetro en cualquier clase de terreno, excepto roca, "
            "incluso  el relleno y compactado de la zanja,  el suministro y montaje "
            "de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, "
            "el transporte y la retirada de los productos al lugar de empleo/n" 
            "Trabajo: Diurno/n"
            "Banda de mantenimiento: i >= 5 horas/n"
            "Condiciones de ejecución: Volumen relevante/n"
        )
    }
]

# =========================
#  Plantilla con PAUTAS
# =========================
PROMPT_TEMPLATE = (
    "Transforma el texto siguiendo estrictamente estas pautas:\n"
    "1) Mantén el significado original y todas las palabras clave.\n"
    "2) Resume el texto sin alterar la naturaleza ni el dimensionamiento del trabajo para que permita identificarse sin ambigüedad.\n\n"
    "Ejemplo:\n"
    "texto a transformar:\n"
    "Zanja para tendido de cables de 1,10 m de profundidad y 0,60 m de anchura máxima, zona de balasto, "
    "ejecutada a mano incluso descerne y entibación de costados y posterior reposición del balasto retirado, "
    "y relleno compactado con material seleccionado procedente de la excavación, cama de arena de 40 cm de espesor, "
    "rejilla de plastico identificadora, carga, descarga y transporte al lugar asignado en el interior de la obra.\n"
    "Trabajo: Cualquier franja horaria\n"
    "Banda de mantenimiento: No necesita intervalo\n"
    "Condiciones de ejecución: Cualquier condición de ejecución.\n\n"
    "texto transformado:\n"
    "ZANJA A MANO DE 1,10 EN BALASTO, "
    "Trabajo: Cualquier franja horaria\n"
    "Banda de mantenimiento: No necesita intervalo\n"
    "Condiciones de ejecución: Cualquier condición de ejecución.\n\n"
    "Devuelve la respuesta ÚNICAMENTE como un JSON con esta estructura EXACTA:\n"
    '{{"item_key":"{item_key}", "texto_alterado":"<texto_transformado>"}}\n'
    "donde <texto_transformado> es el texto final. Máx. {max_palabras} palabras.\n\n"
    "Texto a transformar:\n{texto}"
)

resultados = procesar_items(ITEMS)


=== RESUMEN ===
Procesados: 3 | OK: 3 | Errores: 0


In [21]:
# Convertir a DataFrames
df_items = pd.DataFrame(ITEMS)
df_resultados = pd.DataFrame(resultados)

# Unir por item_key
df_final = pd.merge(df_items, df_resultados, on="item_key", how="inner")

# Visualización simple por consola
print("\n=== RESULTADOS ===")
pd.set_option("display.max_colwidth", None)   # no truncar celdas
pd.set_option("display.width", 0)            # ajuste automático del ancho
pd.set_option("display.max_columns", None)   # mostrar todas las columnas
df_final


=== RESULTADOS ===


,item_key,texto,texto_alterado
0,item1,"Zanja para tendido de cables de 1,10 m de profundidad y 0,60 m de anchura máxima, zona de balasto, ejecutada a mano incluso descerne y entibación de costados y posterior reposición del balasto retirado, y relleno compactado con material seleccionado procedente de la excavación, cama de arena de 40 cm de espesor, rejilla de plástico identificadora, carga, descarga y transporte al lugar asignado en el interior de la obra.\nTrabajo: Cualquier franja horaria\nBanda de mantenimiento: No necesita intervalo\nCondiciones de ejecución: Cualquier condición de ejecución","ZANJA A MANO DE 1,10 EN BALASTO, Trabajo: Cualquier franja horaria Banda de mantenimiento: No necesita intervalo Condiciones de ejecución: Cualquier condición de ejecución."
1,item2,"Zanja para tendido de cables de hasta 0,80 m de profundidad y 0,60 m de anchura máxima, ejecutada por excavación manual en cualquier tipo de terreno excepto roca, y relleno compactado con material seleccionado procedente de la excavación, cama de arena de 20 cm de espesor, rejilla de plastico identificadora, carga, descarga y transporte al lugar asignado en el interior de la obra.\nTrabajo: Diurno\nBanda de mantenimiento: i >= 5 horas\nCondiciones de ejecución: Volumen relevante\n","ZANJA A MANO DE HASTA 0,80 M EN CUALQUIER TERRENO, Trabajo: Diurno Banda de mantenimiento: i >= 5 horas Condiciones de ejecución: Volumen relevante"
2,item3,"Canalización hormigonada de 1 tubos de polietileno libre de halógenos de 110 mm de diámetro en cualquier clase de terreno, excepto roca, incluso el relleno y compactado de la zanja, el suministro y montaje de los tubos y hormigón tipo HE-20 sin vibrar, la prueba de los conductos, el transporte y la retirada de los productos al lugar de empleo/nTrabajo: Diurno/nBanda de mantenimiento: i >= 5 horas/nCondiciones de ejecución: Volumen relevante/n","CANALIZACIÓN HORMIGONADA DE TUBOS DE POLIETILENO DE 110 MM, Trabajo: Diurno Banda de mantenimiento: i >= 5 horas Condiciones de ejecución: Volumen relevante."
